# 05 — Hyperparameter Tuning

**Research**: Analisis Performa Algoritma Machine Learning untuk Klasifikasi Jenis Cyberbullying pada Teks Bahasa Indonesia Menggunakan TF-IDF

**Objective**: Find the best hyperparameter configuration for each algorithm using GridSearchCV.

**Models**:
1. Multinomial Naive Bayes (alpha)
2. Logistic Regression (C, solver, max_iter)
3. Linear SVM (C)

**Search**: GridSearchCV, 5-Fold Stratified CV, f1_macro scoring

**Rules**:
- No evaluation on test set
- No model comparison

---

## 1. Setup

In [1]:
import sys
import os

# Ensure project root is in path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import project modules
from src.config.settings import (
    TUNING_REPORTS_DIR,
    NB_BEST_MODEL_PATH,
    LR_BEST_MODEL_PATH,
    SVM_BEST_MODEL_PATH,
)
from src.config.tuning_config import TuningConfig
from src.utils.load_features import load_features
from src.tuning.tuner import (
    tune_naive_bayes,
    tune_logistic_regression,
    tune_svm,
)
from src.tuning.persistence import (
    save_best_model,
    save_best_parameters,
    save_cv_results,
)
from src.tuning.report_generator import (
    generate_tuning_report,
    export_tuning_statistics,
    save_tuning_report,
)

print('Setup complete.')

Setup complete.


---
## 2. Load Features & Labels

In [2]:
X_train, X_test, y_train, y_test = load_features()

print(f'\nTraining set:  X={X_train.shape}, y={len(y_train):,}')
print(f'Testing set:   X={X_test.shape}, y={len(y_test):,}')
print(f'Labels: {sorted(y_train.unique())}')

✓ Artifacts loaded: X_train=(33244, 22695), X_test=(8312, 22695), vocab=22,695
✓ Features validated: X_train=(33244, 22695), X_test=(8312, 22695), labels=6 classes

Training set:  X=(33244, 22695), y=33,244
Testing set:   X=(8312, 22695), y=8,312
Labels: ['harassment', 'hate_speech', 'insult', 'normal', 'sexually_explicit', 'threat']


---
## 3. Configure Tuning

In [3]:
config = TuningConfig(
    cv=5,
    scoring='f1_macro',
    verbose=1,
    n_jobs=-1,
    refit=True,
    return_train_score=True,
    random_state=42,
)

print('Tuning Configuration:')
for k, v in config.to_dict().items():
    print(f'  {k}: {v}')

Tuning Configuration:
  cv: 5
  scoring: f1_macro
  verbose: 1
  n_jobs: -1
  refit: True
  return_train_score: True
  save_results: True
  random_state: 42


---
## 4. Tune Naive Bayes

In [4]:
nb_result = tune_naive_bayes(X_train, y_train, config)
save_best_model(nb_result['best_model'], NB_BEST_MODEL_PATH, overwrite=True)

  Tuning NaiveBayes...
  Param grid: {'alpha': [0.1, 0.5, 1.0, 2.0, 5.0]}
Fitting 5 folds for each of 5 candidates, totalling 25 fits
  ✓ Best score: 0.4216
  ✓ Best params: {'alpha': 0.1}
  ✓ Duration: 3.44s
  ✓ Best model saved: /home/zapp/Kampus/PM/models/naive_bayes_best.joblib (2.1 MB)


'/home/zapp/Kampus/PM/models/naive_bayes_best.joblib'

---
## 5. Tune Logistic Regression

In [5]:
lr_result = tune_logistic_regression(X_train, y_train, config)
save_best_model(lr_result['best_model'], LR_BEST_MODEL_PATH, overwrite=True)

  Tuning LogisticRegression...
  Param grid: {'C': [0.01, 0.1, 1.0, 10.0], 'solver': ['lbfgs'], 'max_iter': [500, 1000, 2000]}
Fitting 5 folds for each of 12 candidates, totalling 60 fits
  ✓ Best score: 0.4644
  ✓ Best params: {'C': 10.0, 'max_iter': 500, 'solver': 'lbfgs'}
  ✓ Duration: 91.86s
  ✓ Best model saved: /home/zapp/Kampus/PM/models/logistic_regression_best.joblib (1.0 MB)


'/home/zapp/Kampus/PM/models/logistic_regression_best.joblib'

---
## 6. Tune SVM

⚠️ **Note**: SVM tuning searches C values on bare SVC, then wraps the best model in CalibratedClassifierCV for predict_proba support.

In [6]:
svm_result = tune_svm(X_train, y_train, config)
save_best_model(svm_result['best_model'], SVM_BEST_MODEL_PATH, overwrite=True)

  Tuning SVM (Linear)...
  Param grid: {'C': [0.01, 0.1, 1.0, 10.0, 100.0]}
Fitting 5 folds for each of 5 candidates, totalling 25 fits
  Fitting CalibratedClassifierCV with best C=1.0...
  ✓ Best score: 0.4645
  ✓ Best params: {'C': 1.0}
  ✓ Duration: 1485.58s (search=962.84s)
  ✓ Best model saved: /home/zapp/Kampus/PM/models/svm_best.joblib (4.7 MB)


'/home/zapp/Kampus/PM/models/svm_best.joblib'

---
## 7. Save Best Parameters (JSON)

In [7]:
all_results = [nb_result, lr_result, svm_result]

all_best_params = {
    r['model_key']: r['best_params'] for r in all_results
}

save_best_parameters(
    all_best_params,
    os.path.join(TUNING_REPORTS_DIR, 'best_parameters.json'),
    overwrite=True,
)

  ✓ Best parameters saved: /home/zapp/Kampus/PM/reports/tuning/best_parameters.json


'/home/zapp/Kampus/PM/reports/tuning/best_parameters.json'

---
## 8. Save CV Results (CSV)

In [8]:
save_cv_results(
    all_results,
    os.path.join(TUNING_REPORTS_DIR, 'cv_results.csv'),
    overwrite=True,
)

  ✓ CV results saved: /home/zapp/Kampus/PM/reports/tuning/cv_results.csv (22 rows)


'/home/zapp/Kampus/PM/reports/tuning/cv_results.csv'

---
## 9. Tuning Summary

In [9]:
print('Hyperparameter Tuning Summary')
print('=' * 70)
print(f'{"Model":<25} {"Best Score":<15} {"Candidates":<12} {"Duration"}')
print('-' * 70)
for r in all_results:
    print(f'{r["model_name"]:<25} {r["best_score"]:<15.4f} '
          f'{r["n_candidates"]:<12} {r["duration"]:.2f}s')
print('=' * 70)
print()
print('Best Parameters:')
for r in all_results:
    print(f'  {r["model_name"]}: {r["best_params"]}')

Hyperparameter Tuning Summary
Model                     Best Score      Candidates   Duration
----------------------------------------------------------------------
NaiveBayes                0.4216          5            3.44s
LogisticRegression        0.4644          12           91.86s
SVM                       0.4645          5            1485.58s

Best Parameters:
  NaiveBayes: {'alpha': 0.1}
  LogisticRegression: {'C': 10.0, 'max_iter': 500, 'solver': 'lbfgs'}
  SVM: {'C': 1.0}


---
## 10. Generate Reports

In [10]:
print('Generating reports...\n')

# Tuning summary report (markdown)
report_content = generate_tuning_report(all_results, config)
save_tuning_report(
    report_content,
    os.path.join(TUNING_REPORTS_DIR, 'tuning_summary.md'),
)

# Tuning statistics CSV
export_tuning_statistics(
    all_results,
    os.path.join(TUNING_REPORTS_DIR, 'tuning_statistics.csv'),
)

print(f'\n✓ All reports generated.')

Generating reports...

  ✓ Tuning report saved: /home/zapp/Kampus/PM/reports/tuning/tuning_summary.md
  ✓ Tuning statistics exported: /home/zapp/Kampus/PM/reports/tuning/tuning_statistics.csv

✓ All reports generated.


---
## Summary

Hyperparameter tuning is complete. Generated outputs:

**Best Models**:
- `models/naive_bayes_best.joblib`
- `models/logistic_regression_best.joblib`
- `models/svm_best.joblib`

**Reports**:
- `reports/tuning/tuning_summary.md`
- `reports/tuning/tuning_statistics.csv`
- `reports/tuning/best_parameters.json`
- `reports/tuning/cv_results.csv`

**Next Step**: `06_model_evaluation.ipynb` — Model Evaluation